# Pour 100 dollars de PIB américain · *For every 100 dollars of U.S. GDP*

Notebook compagnon du chapitre **9. Comprendre le PIB : la mesure de la richesse produite par un pays** — [lire l'article](https://nmlab.io/ressources/comprendre-le-pib).
Companion notebook to chapter **9. Understanding GDP: The Measure of the Wealth a Country Produces** — [read the article](https://nmlab.io/en/ressources/understanding-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_weights() -> list[float]:
    """Poids moyens 2025 de C, I, G et (X − M) dans le PIB américain, en % (FRED, en direct).
    2025 average weights of C, I, G and net exports in U.S. GDP, live from FRED."""
    gdp = nm.load_fred("GDP")
    g25 = gdp[gdp.index.year == 2025]
    weights = []
    for code in ("PCEC", "GPDI", "GCE", "NETEXP"):
        s = nm.load_fred(code)
        s25 = s[s.index.year == 2025]
        n = min(len(s25), len(g25))
        weights.append(float((s25.values[:n] / g25.values[:n] * 100).mean()))
    return weights

weights = load_weights()


from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Pour 100 dollars de PIB américain",
        sub="poids moyen de chaque poste de la dépense en 2025",
        cats=["Consommation des ménages (C)", "Investissement privé (I)",
              "Dépenses et invest. publics (G)", "Exportations nettes (X − M)"],
        hint="les quatre postes se somment à 100 %",
        note="Sources : BEA via FRED (PCEC, GPDI, GCE, NETEXP, GDP), moyenne des quatre trimestres de 2025."),
    "en": dict(
        title="For every 100 dollars of U.S. GDP",
        sub="average weight of each spending block in 2025",
        cats=["Household consumption (C)", "Private investment (I)",
              "Government purchases (G)", "Net exports (X − M)"],
        hint="the four blocks sum to 100%",
        note="Sources: BEA via FRED (PCEC, GPDI, GCE, NETEXP, GDP), average of the four quarters of 2025."),
}

def pct(value: float, lang: str) -> str:
    """Étiquette de pourcentage à une décimale, localisée."""
    sign = "−" if value < 0 else ""
    body = f"{abs(value):.1f}".replace(".", ",") if lang == "fr" else f"{abs(value):.1f}"
    return f"{sign}{body} %" if lang == "fr" else f"{sign}{body}%"

def build_figure(weights: list[float], lang: str) -> Figure:
    """Barres horizontales : C, I, G en bleu, exportations nettes en rose (négatif)."""
    text = LABELS[lang]
    fig = nm.figure(height_px=988)
    ax = nm.axes(fig, left=0.315, right=0.965, bottom=0.14)
    ax.grid(axis="y", visible=False)
    positions = [3, 2, 1, 0]
    colors = [nm.COLORS["blue"], nm.COLORS["blue"], nm.COLORS["blue"], nm.COLORS["rose"]]
    ax.barh(positions, weights, height=0.62, color=colors, zorder=3)
    ax.axvline(0, color=nm.COLORS["muted"], linewidth=1.4, alpha=0.9)
    ax.set_xlim(-6, 78)
    ax.set_xticks(range(0, 71, 10))
    ax.set_ylim(-0.7, 3.7)
    ax.set_yticks(positions)
    ax.set_yticklabels(text["cats"], fontsize=24, color=nm.COLORS["text"])
    ax.tick_params(axis="y", length=0)
    for pos, value in zip(positions, weights):
        colour = nm.COLORS["rose"] if value < 0 else nm.COLORS["text"]
        ax.text(value + 1.3 if value >= 0 else 1.3, pos, pct(value, lang),
                va="center", ha="left", fontsize=26, fontweight="bold", color=colour)
    ax.text(77, 0.6, text["hint"], ha="right", va="center", fontsize=23,
            fontstyle="italic", color=nm.COLORS["muted"])
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(weights, LANG)